# Feature Engineering — NLP Features
Reads processed_posts from reddit_warehouse.db, computes NLP features (VADER sentiment, NRC emotions, readability) keyed on post_id for the XGBoost model.

- vader_compound: sentiment, -1 to +1
- nrc_*: 8 emotion scores (joy, trust, fear, surprise, sadness, disgust, anger, anticipation)
- readability: Flesch Reading Ease (higher = easier to read); swappable pending Sarah's decision
- empty text -> NaN, text with no emotion words -> 0.0

Final output table: `zoe_nlp_features` keyed on `post_id` (title/body are input only, dropped from output).

In [ ]:
import duckdb


# notebook lives in notebooks/zoe/, db is at project root, so go up two levels
con = duckdb.connect("../../reddit_warehouse.db")

print(con.execute("SHOW TABLES").df())
posts = con.execute("SELECT post_id, title, body FROM processed_posts").df()
print(posts.shape)
posts.head()

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import numpy as np

analyzer = SentimentIntensityAnalyzer()

def get_vader_compound(title, body):
    # combine title + body; this is your swappable text scope
    text = f"{title} {body}".strip()
    if not text:
        return np.nan          # no text at all -> missing, not 0
    return analyzer.polarity_scores(text)["compound"]

posts["vader_compound"] = posts.apply(
    lambda row: get_vader_compound(row["title"], row["body"]), axis=1
)

posts[["title", "body", "vader_compound"]].head(10)

In [ ]:
import urllib.request, os
import numpy as np
from collections import defaultdict

LEX_PATH = "nrc_lexicon.txt"
if not os.path.exists(LEX_PATH):
    url = "https://raw.githubusercontent.com/dinbav/LeXmo/master/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt"
    urllib.request.urlretrieve(url, LEX_PATH)

EMOTIONS = ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]
word_emotions = defaultdict(set)

with open(LEX_PATH) as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) != 3:          # skip headers, blank lines, anything malformed
            continue
        word, emotion, flag = parts
        if flag == "1" and emotion in EMOTIONS:
            word_emotions[word].add(emotion)

print("lexicon loaded, words:", len(word_emotions))

In [ ]:
import re

def get_emotions(title, body):
    text = f"{title} {body}".strip().lower()
    if not text:
        # no text at all -> missing, not 0
        return {f"nrc_{e}": np.nan for e in EMOTIONS}

    words = re.findall(r"[a-z']+", text)   # simple word tokenizer
    if not words:
        # text exists but has no alphabetic tokens (e.g. "156 @ 81") -> no emotion words -> 0.0
        # (consistent with the rule: NaN only when title AND body are both empty)
        return {f"nrc_{e}": 0.0 for e in EMOTIONS}

    # count emotion hits across all words
    counts = {e: 0 for e in EMOTIONS}
    for w in words:
        for emotion in word_emotions.get(w, ()):
            counts[emotion] += 1

    # normalize by total words so long posts don't score higher just for length
    n = len(words)
    return {f"nrc_{e}": counts[e] / n for e in EMOTIONS}

emotion_cols = posts.apply(
    lambda row: get_emotions(row["title"], row["body"]),
    axis=1, result_type="expand"
)
posts = posts.join(emotion_cols)

posts[["title", "nrc_anger", "nrc_joy", "nrc_fear", "nrc_sadness"]].head(10)

In [ ]:
import textstat

# Readability feature: Flesch Reading Ease. Higher = easier to read.
# NOTE: metric is SWAPPABLE pending Sarah's decision. To change it, swap only the
# textstat call below (e.g. textstat.flesch_kincaid_grade / textstat.gunning_fog).
def get_readability(title, body):
    text = f"{title} {body}".strip()
    if not text:
        # defensive empty-text guard: NaN when title AND body are both empty
        # (does not trigger on current data, but keeps the rule consistent with VADER/NRC)
        return np.nan
    return textstat.flesch_reading_ease(text)

posts["readability"] = posts.apply(
    lambda row: get_readability(row["title"], row["body"]), axis=1
)

posts[["title", "readability"]].head(10)

In [ ]:
# ── Assemble final NLP feature table ──────────────────────────────────────
# ONLY these columns; title/body were inputs and are dropped. Keyed on post_id
# so it matches Kristin's LEFT JOIN on post_id.
NLP_COLS = [
    "post_id",
    "vader_compound",
    "nrc_joy", "nrc_trust", "nrc_fear", "nrc_surprise",
    "nrc_sadness", "nrc_disgust", "nrc_anger", "nrc_anticipation",
    "readability",
]
zoe_nlp_features = posts[NLP_COLS].copy()

# Save locally (data artifact; gitignored, never committed)
zoe_nlp_features.to_parquet("zoe_nlp_features.parquet", index=False)

# Also persist into the warehouse so the Phase 3 join can read it from DuckDB
con.register("zoe_nlp_df", zoe_nlp_features)
con.execute("CREATE OR REPLACE TABLE zoe_nlp_features AS SELECT * FROM zoe_nlp_df")
con.unregister("zoe_nlp_df")
db_rows = con.execute("SELECT COUNT(*) FROM zoe_nlp_features").fetchone()[0]
print(f"wrote table zoe_nlp_features to reddit_warehouse.db ({db_rows} rows)")

print("\nshape:", zoe_nlp_features.shape)
print("\ndtypes:")
print(zoe_nlp_features.dtypes)
print("\nhead(10):")
print(zoe_nlp_features.head(10).to_string())

In [ ]:
# ── Sanity checks ─────────────────────────────────────────────────────────
print("null counts per column:")
print(zoe_nlp_features.isna().sum())

print("\nreadability distribution (higher = easier to read):")
print(zoe_nlp_features["readability"].describe())

# Emotional posts should surface non-trivial emotion scores
print("\nTop 5 posts by nrc_anger (should read as angry/negative):")
print(
    posts.sort_values("nrc_anger", ascending=False)
    .loc[:, ["title", "vader_compound", "nrc_anger", "nrc_joy", "readability"]]
    .head(5)
    .to_string()
)

print("\nTop 5 posts by nrc_joy (should read as positive/upbeat):")
print(
    posts.sort_values("nrc_joy", ascending=False)
    .loc[:, ["title", "vader_compound", "nrc_joy", "nrc_anger", "readability"]]
    .head(5)
    .to_string()
)